<a href="https://colab.research.google.com/github/harshit-techy-debug/aitoolsprctmini/blob/main/aitoolpract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%writefile app.py
import streamlit as st
import os
from pypdf import PdfReader
from groq import Groq
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# --- Page Configuration ---
st.set_page_config(page_title="TalkPDF", page_icon="📄", layout="wide")

# --- Custom CSS ---
st.markdown("""
<style>
    [data-testid="stSidebar"] { background-color: #0E1117; color: white; }
    header {visibility: hidden;}
</style>
""", unsafe_allow_html=True)

# --- Session State ---
if "messages" not in st.session_state:
    st.session_state.messages = [{"role": "assistant", "content": "How can I help you today?"}]
if "chunks" not in st.session_state:
    st.session_state.chunks = []
if "pdf_name" not in st.session_state:
    st.session_state.pdf_name = None

# --- Sidebar ---
with st.sidebar:
    st.title("TalkPDF")

    api_key = st.text_input("Groq API Key", type="password", help="Get one at console.groq.com")

    st.divider()

    uploaded_file = st.file_uploader("Upload PDF", type="pdf")

    if uploaded_file is not None:
        if st.session_state.pdf_name != uploaded_file.name:
            with st.spinner("Processing PDF..."):
                reader = PdfReader(uploaded_file)
                text = ""
                for page in reader.pages:
                    if page.extract_text():
                        text += page.extract_text() + "\n"

                splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
                st.session_state.chunks = splitter.split_text(text)
                st.session_state.pdf_name = uploaded_file.name
                st.session_state.messages = [{"role": "assistant", "content": f"I've loaded **{uploaded_file.name}**. Ask me anything!"}]
            st.success("Ready!")

    st.divider()
    if st.session_state.pdf_name:
        st.info(f"📄 {st.session_state.pdf_name}")

# --- Main Layout ---
col1, col2 = st.columns([1, 1], gap="large")

# Left: Doc Viewer
with col1:
    st.subheader("📄 Document Viewer")
    st.divider()
    if st.session_state.chunks:
        doc_container = st.container(height=600)
        with doc_container:
            preview_text = "\n\n".join(st.session_state.chunks[:5])
            st.text(preview_text + "\n\n[Document continues...]")
    else:
        st.info("Upload a PDF to view it here.")

# Right: Chat
with col2:
    st.subheader("💬 Chat")
    st.divider()

    chat_container = st.container(height=500)
    with chat_container:
        for msg in st.session_state.messages:
            with st.chat_message(msg["role"]):
                st.markdown(msg["content"])

    if prompt := st.chat_input("Ask a question..."):
        if not api_key:
            st.error("Please enter your Groq API Key.")
        elif not st.session_state.chunks:
            st.error("Please upload a PDF first.")
        else:
            st.session_state.messages.append({"role": "user", "content": prompt})
            with chat_container:
                with st.chat_message("user"):
                    st.markdown(prompt)

            with chat_container:
                with st.chat_message("assistant"):
                    with st.spinner("Thinking..."):
                        try:
                            # RAG Context
                            vectorizer = TfidfVectorizer(stop_words="english")
                            tfidf_matrix = vectorizer.fit_transform(st.session_state.chunks)
                            query_vec = vectorizer.transform([prompt])
                            similarities = cosine_similarity(query_vec, tfidf_matrix).flatten()
                            top_idx = similarities.argsort()[-3:][::-1]
                            context = "\n\n".join([st.session_state.chunks[i] for i in top_idx])

                            # Groq Call - UPDATED MODEL ID
                            client = Groq(api_key=api_key)
                            system_prompt = f"Answer using ONLY this context:\n{context}"

                            response = client.chat.completions.create(
                                messages=[
                                    {"role": "system", "content": system_prompt},
                                    {"role": "user", "content": prompt}
                                ],
                                model="openai/gpt-oss-20b",  # <-- THIS IS THE FIX
                                temperature=0.2
                            )
                            ans = response.choices[0].message.content
                            st.markdown(ans)
                            st.session_state.messages.append({"role": "assistant", "content": ans})
                        except Exception as e:
                            st.error(f"Error: {e}")

Overwriting app.py


In [ ]:
# 1. Download Cloudflare Tunnel (if it's not downloaded yet)
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64

# 2. Start Streamlit in the background
!streamlit run app.py --server.port 8501 &>/content/logs.txt &

# 3. Start the tunnel
import time
time.sleep(3) # Give Streamlit a second to start

print("\n" + "="*60)
print("LOOK FOR THE LINK ENDING IN '.trycloudflare.com' BELOW")
print("="*60 + "\n")

!./cloudflared-linux-amd64 tunnel --url http://localhost:8501


LOOK FOR THE LINK ENDING IN '.trycloudflare.com' BELOW

2026-10-30T18:23:38Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-10-30T18:23:38Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-10-30T18:23:42Z INF +--------------------------------------------------------------------------------------------+
2026-10-30T18:23:42Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-10-30T18:23:42Z INF